In [90]:
import pandas as pd
import numpy as np
import yfinance as yf

def download_prices(ticker, start_date, end_date):
    prices = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)["Close"]
    return prices

prices = download_prices(["SPY", "QQQ", "GLD"], "2015-01-01", "2026-01-01") 
market_data = pd.DataFrame(prices) # déja un df
# print(market_data.head())
# print(market_data.shape)
# print(market_data.columns)
# print(market_data.dtypes)
# print(market_data.index)
# print(type(market_data))
# print(type(market_data.index))
# print(type(market_data.columns))
# print(market_data.isna().sum())

market_data_reset = market_data.reset_index() # on passe l'index date en colonnee et on crée une colonne d'index de 0 à n avec n le nb d'observations -1 télécharger.
# print(market_data_reset.head())
# print(market_data_reset.columns)
# print(market_data_reset.shape)

market_data_long = market_data_reset.melt(id_vars="Date", var_name="Ticker", value_name="Close")
# print(market_data_long.head())
# print(market_data_long.shape)
# print(market_data_long.isna().sum())
# print(market_data_long.duplicated().sum())
# print(market_data_long.duplicated(subset=["Date", "Ticker"]).sum())

mean_prices_by_ticker = market_data_long.groupby("Ticker")["Close"].mean()
# print(mean_prices_by_ticker)

market_data_sorted = market_data_long.copy()
market_data_sorted = market_data_sorted.sort_values(by=["Ticker","Date"])
# print(market_data_sorted.head())
market_data_sorted["Returns"] = market_data_sorted.groupby("Ticker")["Close"].pct_change()
# print(market_data_sorted.head())
# print(market_data_sorted.isna().sum())

market_data_mean_returns = market_data_sorted.groupby("Ticker")["Returns"].mean()
market_data_volatility = market_data_sorted.groupby("Ticker")["Returns"].std()
# print((market_data_mean_returns * 100).round(2))
# print((market_data_volatility * 100).round(2))

asset_statistics = market_data_sorted.groupby("Ticker")["Returns"].agg(["mean", "std", "count"])
print(asset_statistics)

spy_df = market_data_long[market_data_long["Ticker"] == "SPY"].copy()
spy_df = spy_df[["Date", "Close"]]
spy_df = spy_df.rename(columns={"Close" : "SPY"})
print(spy_df.head())

gld_df = market_data_long[market_data_long["Ticker"] == "GLD"].copy()
gld_df = gld_df[["Date", "Close"]]
gld_df = gld_df.rename(columns={"Close" : "GLD"})
print(gld_df.head())

merged_prices = pd.merge(spy_df, gld_df, on="Date", how="inner")
print(merged_prices.shape)
print(merged_prices.head())
merged_prices_outer = pd.merge(spy_df, gld_df, on="Date", how="outer")
print(merged_prices_outer.shape)
print(merged_prices_outer.isna().sum())
print(merged_prices.shape[0] - merged_prices_outer.shape[0])

            mean       std  count
Ticker                           
GLD     0.000494  0.009290   2765
QQQ     0.000772  0.013847   2765
SPY     0.000564  0.011208   2765
           Date         SPY
5532 2015-01-02  169.267578
5533 2015-01-05  166.210587
5534 2015-01-06  164.645111
5535 2015-01-07  166.696777
5536 2015-01-08  169.654846
        Date         GLD
0 2015-01-02  114.080002
1 2015-01-05  115.800003
2 2015-01-06  117.120003
3 2015-01-07  116.430000
4 2015-01-08  115.940002
(2766, 3)
        Date         SPY         GLD
0 2015-01-02  169.267578  114.080002
1 2015-01-05  166.210587  115.800003
2 2015-01-06  164.645111  117.120003
3 2015-01-07  166.696777  116.430000
4 2015-01-08  169.654846  115.940002
(2766, 3)
Date    0
SPY     0
GLD     0
dtype: int64
0
